# Data Analysis For All Weather Stations From IMS

In [6]:
import pandas as pd
from weather_engine.database import engine

CORE_FEATURES = ['rain', 'ws', 'stdwd', 'td', 'rh', 'tdmax', 'tdmin', 'wd']

missingness_cols_sql = ', '.join([
    f"ROUND(100.0 * SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS {col}_missingness"
    for col in CORE_FEATURES
])

query = f"""
    SELECT 
        station_id,
        COUNT(*) as total_rows,
        {missingness_cols_sql}
    FROM raw_station_data
    GROUP BY station_id
    ORDER BY total_rows ASC
"""

final_report_df = pd.read_sql(query, engine)

# Pull coordinates and join
coords_df = pd.read_sql("SELECT station_id, latitude, longitude FROM station_metadata", engine)
station_coords_df = final_report_df.merge(coords_df, on='station_id', how='left')

missingness_cols = [c for c in final_report_df.columns if c.endswith('_missingness')]
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    styled_df = final_report_df.style.background_gradient(
        cmap='RdYlGn_r',
        subset=missingness_cols,
        vmin=0,
        vmax=100
    )
    display(styled_df)

,station_id,total_rows,rain_missingness,ws_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,wd_missingness
0,504,6091,0.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,502,76903,0.000000,0.010000,0.010000,0.010000,0.020000,0.010000,0.010000,0.010000
2,500,95231,0.000000,5.540000,5.540000,0.020000,0.010000,0.010000,0.020000,5.540000
3,32,102019,100.000000,100.000000,100.000000,0.180000,0.170000,100.000000,100.000000,100.000000
4,411,160303,0.000000,1.870000,1.880000,0.030000,0.030000,0.030000,0.030000,1.870000
5,115,220643,74.140000,0.010000,0.010000,0.000000,0.000000,0.000000,0.010000,0.010000
6,60,267080,100.000000,100.000000,100.000000,90.790000,100.000000,100.000000,100.000000,100.000000
7,263,272305,0.270000,100.000000,100.000000,0.430000,0.230000,0.230000,0.230000,100.000000
8,106,276892,3.950000,0.050000,0.060000,0.080000,0.060000,0.090000,0.080000,0.100000
9,241,281112,2.300000,0.010000,0.010000,1.340000,1.340000,1.340000,1.340000,0.010000


## Filtering out problematic stations that miss alot of sensors

In [7]:
cols_to_ignore = ['elevation_missingness', 
                    'ws1mm_missingness', 'ws10mm_missingness',
                    'timestamp_missingness', 'station_id_missingness', 
                    'latitude_missingness', 'longitude_missingness']

cols_to_check = [c for c in final_report_df.columns 
                 if c.endswith('_missingness') 
                 and c not in cols_to_ignore]

print(cols_to_check)

stations_to_leave = []

for idx, row in final_report_df.iterrows():
    for col in cols_to_check:
        if row[col] >= 30.0:
            stations_to_leave.append(row['station_id'])
            break

df_bad_stations = final_report_df[final_report_df['station_id'].isin(stations_to_leave)]
print(len(df_bad_stations))
df_bad_stations
            

['rain_missingness', 'ws_missingness', 'stdwd_missingness', 'td_missingness', 'rh_missingness', 'tdmax_missingness', 'tdmin_missingness', 'wd_missingness']
22


,station_id,total_rows,rain_missingness,ws_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,wd_missingness
0,504,6091,0.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
3,32,102019,100.00,100.00,100.00,0.18,0.17,100.00,100.00,100.00
5,115,220643,74.14,0.01,0.01,0.00,0.00,0.00,0.01,0.01
6,60,267080,100.00,100.00,100.00,90.79,100.00,100.00,100.00,100.00
7,263,272305,0.27,100.00,100.00,0.43,0.23,0.23,0.23,100.00
11,236,288889,3.50,100.00,100.00,0.03,0.05,0.04,0.02,100.00
12,64,292674,0.00,71.14,71.14,0.03,0.04,0.03,0.03,71.14
14,99,307239,0.44,100.00,100.00,0.02,0.01,0.02,0.02,100.00
27,62,314372,0.37,41.10,41.10,0.10,0.49,0.10,0.10,41.10
29,75,314456,0.39,100.00,100.00,0.02,0.48,0.01,0.02,100.00


In [8]:
# How many stations have ANY column over 30%?
print(f"Total stations: {len(final_report_df)}")
print(f"Stations with 30%+ missingness in any column: {len(stations_to_leave)}")

# Which columns are causing the most flags?
for col in cols_to_check:
    over_30 = (final_report_df[col] >= 30.0).sum()
    if over_30 > 0:
        print(f"{col}: {over_30} stations over 30%")

Total stations: 88
Stations with 30%+ missingness in any column: 22
rain_missingness: 4 stations over 30%
ws_missingness: 21 stations over 30%
stdwd_missingness: 21 stations over 30%
td_missingness: 3 stations over 30%
rh_missingness: 3 stations over 30%
tdmax_missingness: 5 stations over 30%
tdmin_missingness: 5 stations over 30%
wd_missingness: 21 stations over 30%


In [9]:
df_survived_stations = final_report_df[~final_report_df['station_id'].isin(df_bad_stations['station_id'])]
print(len(df_survived_stations))

good_ids = df_survived_stations['station_id']
bad_ids = df_bad_stations['station_id']
shared = good_ids[good_ids.isin(bad_ids)]

if len(shared) > 0:
    print('Detected duplicated station in good and bad dataframes')
else:
    print('No duplicate stations were found in good and bad dataframes')

display(df_survived_stations.head())
display(df_bad_stations.head())

66
No duplicate stations were found in good and bad dataframes


,station_id,total_rows,rain_missingness,ws_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,wd_missingness
1,502,76903,0.00,0.01,0.01,0.01,0.02,0.01,0.01,0.01
2,500,95231,0.00,5.54,5.54,0.02,0.01,0.01,0.02,5.54
4,411,160303,0.00,1.87,1.88,0.03,0.03,0.03,0.03,1.87
8,106,276892,3.95,0.05,0.06,0.08,0.06,0.09,0.08,0.10
9,241,281112,2.30,0.01,0.01,1.34,1.34,1.34,1.34,0.01


,station_id,total_rows,rain_missingness,ws_missingness,stdwd_missingness,td_missingness,rh_missingness,tdmax_missingness,tdmin_missingness,wd_missingness
0,504,6091,0.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
3,32,102019,100.00,100.00,100.00,0.18,0.17,100.00,100.00,100.00
5,115,220643,74.14,0.01,0.01,0.00,0.00,0.00,0.01,0.01
6,60,267080,100.00,100.00,100.00,90.79,100.00,100.00,100.00,100.00
7,263,272305,0.27,100.00,100.00,0.43,0.23,0.23,0.23,100.00


In [13]:
import folium
import pandas as pd

coords = station_coords_df.copy()
coords['station_id'] = coords['station_id'].astype(float).astype(int).astype(str)

healthy_ids = set(df_survived_stations['station_id'].astype(str))
bad_ids = set(df_bad_stations['station_id'].astype(str))

healthy_coords = coords[coords['station_id'].isin(healthy_ids)]
bad_coords = coords[coords['station_id'].isin(bad_ids)]

m = folium.Map(tiles='cartodbdark_matter', location=[31.5, 34.8], zoom_start=8)

for idx, row in healthy_coords.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='green',
        fill=True,
        fill_opacity=0.8,
        popup=f"ID: {row['station_id']}"
    ).add_to(m)

for idx, row in bad_coords.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='red',
        fill=True,
        fill_opacity=0.8,
        popup=f"ID: {row['station_id']}"
    ).add_to(m)

m